# Used Car Price Prediction

## 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

## 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline


In [2]:
# Keep the CSV file in the same folder as this notebook.
df = pd.read_csv("cardekho_imputated.csv", index_col=0)


In [3]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Data Cleaning
### Handling Missing values

* Handling Missing values 
* Handling Duplicates
* Check data type
* Understand the dataset

In [4]:
# Check missing values and duplicate rows
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())


car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

Duplicate rows: 167


In [5]:
# Remove columns that do not provide useful information for this model.
df = df.drop(columns=["car_name", "brand"], errors="ignore")


In [6]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [7]:
df['model'].unique()

<StringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [35]:
print(sorted(df["model"].dropna().unique()))

['3', '5', '6', '7', 'A4', 'A6', 'A8', 'Alto', 'Altroz', 'Alturas', 'Amaze', 'Aspire', 'Aura', 'Baleno', 'Bolero', 'C', 'C-Class', 'CLS', 'CR', 'CR-V', 'Camry', 'Carnival', 'Cayenne', 'Celerio', 'Ciaz', 'City', 'Civic', 'Compass', 'Continental', 'Cooper', 'Creta', 'D-Max', 'Duster', 'Dzire LXI', 'Dzire VXI', 'Dzire ZXI', 'E-Class', 'ES', 'Ecosport', 'Eeco', 'Elantra', 'Endeavour', 'Ertiga', 'F-PACE', 'Figo', 'Fortuner', 'Freestyle', 'GL-Class', 'GLS', 'GO', 'GTC4Lusso', 'Ghibli', 'Ghost', 'Glanza', 'Grand', 'Gurkha', 'Harrier', 'Hector', 'Hexa', 'Ignis', 'Innova', 'Jazz', 'KUV', 'KUV100', 'KWID', 'Kicks', 'MUX', 'Macan', 'Marazzo', 'NX', 'Nexon', 'Octavia', 'Panamera', 'Polo', 'Q7', 'Quattroporte', 'RX', 'Rapid', 'RediGO', 'Rover', 'S-Class', 'S-Presso', 'S90', 'Safari', 'Santro', 'Scorpio', 'Seltos', 'Superb', 'Swift', 'Swift Dzire', 'Thar', 'Tiago', 'Tigor', 'Triber', 'Tucson', 'Vento', 'Venue', 'Verna', 'Vitara', 'WR-V', 'Wagon R', 'Wrangler', 'X-Trail', 'X1', 'X3', 'X4', 'X5', 'XC'

In [8]:
## Getting All Different Types of Features

# 1. Numerical features
num_features = df.select_dtypes(include=['number']).columns.tolist()
print('Num of Numerical Features:', len(num_features))

# 2. Categorical features
cat_features = df.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()
print('Num of Categorical Features:', len(cat_features))

# 3. Discrete numerical features (25 or fewer unique values)
discrete_features = [
    feature for feature in num_features
    if df[feature].nunique() <= 25
]
print('Num of Discrete Features:', len(discrete_features))

# 4. Continuous numerical features
continuous_features = [
    feature for feature in num_features
    if feature not in discrete_features
]
print('Num of Continuous Features:', len(continuous_features))

Num of Numerical Features: 7
Num of Categorical Features: 4
Num of Discrete Features: 2
Num of Continuous Features: 5


In [10]:
# Separate features and target. Keep X as a DataFrame so column names remain available.
from sklearn.model_selection import train_test_split

if "selling_price" not in df.columns:
    raise KeyError("The target column 'selling_price' was not found in the dataset.")

X = df.drop(columns=["selling_price"])
y = df["selling_price"]

# Split before fitting any preprocessing steps to prevent data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [11]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


## Feature Encoding and Scaling
**One Hot Encoding for Columns which had lesser unique values and not ordinal**
* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [12]:
len(df['model'].unique())

120

In [13]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [15]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [16]:
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

In [17]:
# Preprocess categorical and numerical features.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", StandardScaler(), numerical_features),
    ],
    remainder="drop",
)


In [18]:
# Fit only on training features; apply the learned transformations to test features.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [19]:
print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


Processed training shape: (12328, 134)
Processed test shape: (3083, 134)


In [20]:
# Data has already been split before preprocessing.
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((12328, 10), (3083, 10), (12328,), (3083,))

In [21]:
X_train.head()


,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
14238,XC60,7,70252,Dealer,Diesel,Automatic,11.20,2400,215.0,5
1731,Tiago,2,10000,Individual,Petrol,Manual,23.84,1199,84.0,5
13218,CLS,2,6000,Dealer,Diesel,Automatic,19.00,1950,241.3,5
403,City,7,63000,Dealer,Petrol,Manual,17.80,1497,117.3,5
13550,i10,10,80292,Dealer,Petrol,Manual,20.36,1197,78.9,5


## Model Training And Model Selection

In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [23]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square


In [32]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(max_iter=10000),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_train_pred = model.predict(X_train_processed)
    y_test_pred = model.predict(X_test_processed)

    train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    print(name)
    print("Training set")
    print(f"- RMSE: {train_rmse:.4f}")
    print(f"- MAE:  {train_mae:.4f}")
    print(f"- R2:   {train_r2:.4f}")
    print("Test set")
    print(f"- RMSE: {test_rmse:.4f}")
    print(f"- MAE:  {test_mae:.4f}")
    print(f"- R2:   {test_r2:.4f}")
    print("=" * 35)


Linear Regression
Training set
- RMSE: 344702.2284
- MAE:  167199.9031
- R2:   0.8535
Test set
- RMSE: 395305.0104
- MAE:  179167.9557
- R2:   0.7924
Lasso
Training set
- RMSE: 344703.3506
- MAE:  167201.6356
- R2:   0.8535
Test set
- RMSE: 404061.9085
- MAE:  180328.3799
- R2:   0.7831
Ridge
Training set
- RMSE: 395261.0325
- MAE:  186299.5436
- R2:   0.8074
Test set
- RMSE: 408321.7740
- MAE:  197057.5408
- R2:   0.7785
K-Neighbors Regressor
Training set
- RMSE: 317857.9142
- MAE:  88064.5685
- R2:   0.8754
Test set
- RMSE: 268932.8317
- MAE:  111152.1002
- R2:   0.9039
Decision Tree
Training set
- RMSE: 20797.2352
- MAE:  5164.8199
- R2:   0.9995
Test set
- RMSE: 295269.7822
- MAE:  123092.7857
- R2:   0.8842
Random Forest Regressor
Training set
- RMSE: 151514.2203
- MAE:  39331.2802
- R2:   0.9717
Test set
- RMSE: 217670.5768
- MAE:  99403.8604
- R2:   0.9371


In [25]:
# Example hyperparameter search spaces.
knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}
rf_params = {
    "max_depth": [5, 8, 10, 15, None],
    "max_features": [1.0, "sqrt", 0.5],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500],
}


In [26]:
randomcv_models = [
    ("KNN", KNeighborsRegressor(), knn_params),
    ("RF", RandomForestRegressor(random_state=42, n_jobs=-1), rf_params),
]


In [27]:

from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=20,
        cv=3,
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train_processed, y_train)
    model_param[name] = search.best_params_
    print(name, search.best_params_)


Fitting 3 folds for each of 6 candidates, totalling 18 fits
KNN {'n_neighbors': 10}
Fitting 3 folds for each of 20 candidates, totalling 60 fits
RF {'n_estimators': 200, 'min_samples_split': 2, 'max_features': 1.0, 'max_depth': None}


In [28]:
# Compare selected final models on the same processed train/test data.
final_models = {
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=200, min_samples_split=2, max_features=1.0,
        random_state=42, n_jobs=-1
    ),
    "K-Neighbors Regressor": KNeighborsRegressor(n_neighbors=10, n_jobs=-1),
}

for name, model in final_models.items():
    model.fit(X_train_processed, y_train)
    train_pred = model.predict(X_train_processed)
    test_pred = model.predict(X_test_processed)
    train_mae, train_rmse, train_r2 = evaluate_model(y_train, train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, test_pred)
    print(name)
    print(f"Train: RMSE={train_rmse:.4f}, MAE={train_mae:.4f}, R2={train_r2:.4f}")
    print(f"Test:  RMSE={test_rmse:.4f}, MAE={test_mae:.4f}, R2={test_r2:.4f}")
    print("=" * 35)


Random Forest Regressor
Train: RMSE=140622.7509, MAE=38997.7875, R2=0.9756
Test:  RMSE=218993.1305, MAE=99126.2499, R2=0.9363
K-Neighbors Regressor
Train: RMSE=355121.8862, MAE=98663.1733, R2=0.8445
Test:  RMSE=254103.2537, MAE=113728.3247, R2=0.9142


##Checking the new data


In [29]:
# Example car record. Use a model name that exists in the dataset.
new_car = pd.DataFrame({
    "model": ["Swift"],
    "vehicle_age": [6],
    "km_driven": [45000],
    "seller_type": ["Individual"],
    "fuel_type": ["Petrol"],
    "transmission_type": ["Manual"],
    "mileage": [21.0],
    "engine": [1197],
    "max_power": [81.8],
    "seats": [5],
})


In [ ]:
# Use a fitted model from the final model comparison.
best_model = final_models["Random Forest Regressor"]
new_car_processed = preprocessor.transform(new_car)
predicted_price = best_model.predict(new_car_processed)[0]
print(f"Predicted selling price: {predicted_price:,.0f}")


Predicted selling price: 442,575


In [ ]:
import pickle

# Save the fitted model and fitted preprocessor
with open("random_forest_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

with open("preprocessor.pkl", "wb") as file:
    pickle.dump(preprocessor, file)

In [34]:
import pickle

# Save the fitted Random Forest model
with open("random_forest_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

# Save the fitted preprocessor
with open("preprocessor.pkl", "wb") as file:
    pickle.dump(preprocessor, file)

print("Saved random_forest_model.pkl and preprocessor.pkl")

Saved random_forest_model.pkl and preprocessor.pkl
